# Studi Komparatif Model Generatif: Vanilla AE, Sparse AE, VAE, Vanilla GAN, DCGAN, DDPM
Tugas Take-Home Deep Learning Sesi 07 — Fashion-MNIST



## 0. Setup

In [1]:
!nvidia-smi

Mon Nov 23 09:14:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   70C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install -q torchmetrics torch-fidelity

In [4]:
import os

PROJECT_DIR = "/content/drive/MyDrive/DL_Tugas_Sesi07"
QUICK_TEST = False   # True = uji cepat (±5 menit). Setelah sukses: ubah ke False, Restart session, Run all.

os.makedirs(f"{PROJECT_DIR}/src/models", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/notebook", exist_ok=True)
os.chdir(PROJECT_DIR)
os.environ["PROJECT_DIR"] = PROJECT_DIR
os.environ["QUICK_TEST"] = "1" if QUICK_TEST else "0"
print("Folder proyek:", os.getcwd(), "| QUICK_TEST:", QUICK_TEST)

Folder proyek: /content/drive/MyDrive/DL_Tugas_Sesi07 | QUICK_TEST: False


## 1. Kode sumber (ditulis ke Drive sebagai skrip modular)

In [5]:
%%writefile requirements.txt
torch>=2.3
torchvision>=0.18
torchmetrics>=1.4
torch-fidelity>=0.3
numpy
pandas
matplotlib
scipy

Overwriting requirements.txt


In [6]:
%%writefile README.md
# Studi Komparatif Model Generatif: AE, Sparse AE, VAE, GAN, DCGAN, DDPM

Tugas Take-Home Deep Learning Sesi 07 (BINUS). Dataset: Fashion-MNIST (split 80/10/10, seed 42).

## Struktur
```
src/
  config.py        hyperparameter
  data.py          dataloader + split 80/10/10
  models/          mlp.py, autoencoders.py, gans.py, ddpm.py
  train.py         training semua model (checkpoint per epoch, bisa resume)
  experiments.py   eksperimen batasan MLP Classifier (Tahap 2-5) + interpolasi laten
  evaluate.py      MSE, PSNR, SSIM, FID, latency, training time
  figures.py       Figur 1-5
  run.py           menjalankan seluruh pipeline
notebook/DL_Tugas_Sesi07.ipynb
```

## Menjalankan
```bash
pip install -r requirements.txt
python -m src.run --part A   # MLP + frozen decoder, Vanilla AE, Sparse AE, VAE, Vanilla GAN, DCGAN
python -m src.run --part B   # DDPM (training + sampling)
python -m src.run --part C   # evaluasi + 5 figur
```
Uji cepat pipeline: `QUICK_TEST=1 python -m src.run --part all` (hasil di `outputs_test/`).

Hasil: `outputs/results/` (tabel CSV/Markdown), `outputs/figures/` (Figur 1-5), `outputs/checkpoints/`.
Semua eksperimen dijalankan di Google Colab (GPU Tesla T4).

Overwriting README.md


In [7]:
%%writefile src/__init__.py
"""Studi komparatif model generatif: Vanilla AE, Sparse AE, VAE, Vanilla GAN, DCGAN, DDPM + MLP Classifier."""

Overwriting src/__init__.py


In [8]:
%%writefile src/config.py
"""Konfigurasi global eksperimen."""
import os

PROJECT_DIR = os.environ.get("PROJECT_DIR", os.path.abspath("."))
QUICK_TEST = os.environ.get("QUICK_TEST", "0") == "1"   # 1 = uji cepat pipeline
DATA_DIR = os.environ.get(
    "DATA_DIR", "/content/data" if os.path.isdir("/content") else os.path.join(PROJECT_DIR, "data")
)

# ---------------------------------------------------------------- Umum
SEED = 42
BATCH_SIZE = 128            # sesuai aturan: 128 atau 256
SPLIT = (0.8, 0.1, 0.1)     # train / val / test
IMG_SHAPE = (1, 28, 28)
IMG_DIM = 28 * 28
CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

# ---------------------------------------------------------------- Arsitektur
HIDDEN_DIM = 256            # 784 -> 256 -> 64 (sama untuk MLP Classifier dan semua AE)
LATENT_DIM = 64             # dimensi laten AE / Sparse AE / VAE / z_MLP
NOISE_DIM = 100             # dimensi prior noise Vanilla GAN dan DCGAN
SPARSE_LAMBDA = 1e-4        # lambda pada penalti L1 ||h||_1 Sparse AE

# DDPM
DDPM_T = 1000
DDPM_BETA_START = 1e-4
DDPM_BETA_END = 0.02
DDPM_BASE_CH = 64
EMA_DECAY = 0.999
USE_AMP = True              # mixed precision (hanya aktif bila GPU tersedia)

# ---------------------------------------------------------------- Training
EPOCHS = dict(mlp=20, frozen_decoder=15, vanilla_ae=20, sparse_ae=20, vae=20,
              vanilla_gan=50, dcgan=20, ddpm=20)
LR = dict(mlp=1e-3, frozen_decoder=1e-3, vanilla_ae=1e-3, sparse_ae=1e-3, vae=1e-3,
          vanilla_gan=2e-4, dcgan=2e-4, ddpm=2e-4)
GAN_BETAS = (0.5, 0.999)
MLP_MIN_ACC = 0.88          # syarat tugas: akurasi test MLP >= 88%

# ---------------------------------------------------------------- Evaluasi
N_FID = 2000                # jumlah sampel real & sintetis untuk FID (sama untuk semua model)
FID_BATCH = 250
LATENCY_BATCH = 64          # latency diukur per batch 64 sampel
LATENCY_RUNS = 10
LATENCY_RUNS_DDPM = 3       # DDPM 1000 langkah -> cukup 3 kali ulang
INTERP_ALPHAS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
INTERP_CLASS_A = 0          # T-shirt/top
INTERP_CLASS_B = 9          # Ankle boot (kelas yang kontras)
DDPM_TRAJ_STEPS = [1000, 800, 600, 400, 300, 200, 100, 50, 0]

# ---------------------------------------------------------------- Mode uji cepat
if QUICK_TEST:
    EPOCHS = {k: 1 for k in EPOCHS}
    DDPM_T = 20
    DDPM_TRAJ_STEPS = [20, 15, 10, 5, 0]
    N_FID = 300
    LATENCY_RUNS = 2
    LATENCY_RUNS_DDPM = 1

# ---------------------------------------------------------------- Folder output
OUT_DIR = os.path.join(PROJECT_DIR, "outputs_test" if QUICK_TEST else "outputs")
CKPT_DIR = os.path.join(OUT_DIR, "checkpoints")
LOG_DIR = os.path.join(OUT_DIR, "logs")
FIG_DIR = os.path.join(OUT_DIR, "figures")
RESULT_DIR = os.path.join(OUT_DIR, "results")
SAMPLE_DIR = os.path.join(OUT_DIR, "samples")

Overwriting src/config.py


In [9]:
%%writefile src/utils.py
"""Utilitas umum: seed, device, checkpoint, timer, konversi rentang piksel."""
import json
import os
import random
import time

import numpy as np
import torch

from . import config as C


def set_seed(seed=None):
    """Menetapkan random seed agar eksperimen reprodusibel."""
    seed = C.SEED if seed is None else seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def ensure_dirs():
    for d in [C.OUT_DIR, C.CKPT_DIR, C.LOG_DIR, C.FIG_DIR, C.RESULT_DIR, C.SAMPLE_DIR, C.DATA_DIR]:
        os.makedirs(d, exist_ok=True)


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def load_json(path):
    with open(path) as f:
        return json.load(f)


def ckpt_path(name):
    return os.path.join(C.CKPT_DIR, f"{name}.pt")


def save_ckpt(name, state):
    """Simpan checkpoint secara atomik (aman bila Colab terputus saat menyimpan)."""
    path = ckpt_path(name)
    tmp = path + ".tmp"
    torch.save(state, tmp)
    os.replace(tmp, path)


def load_ckpt(name, device):
    path = ckpt_path(name)
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location=device, weights_only=False)


def sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize()


class Timer:
    """Timer akurat untuk GPU (memanggil cuda.synchronize sebelum/sesudah)."""

    def __init__(self, device):
        self.device = device

    def __enter__(self):
        sync(self.device)
        self.t0 = time.perf_counter()
        return self

    def __exit__(self, *args):
        sync(self.device)
        self.elapsed = time.perf_counter() - self.t0


def to_pm1(x):
    """[0, 1] -> [-1, 1] (input GAN dan DDPM)."""
    return x * 2.0 - 1.0


def to_01(x):
    """[-1, 1] -> [0, 1] (untuk metrik & visualisasi)."""
    return ((x + 1.0) / 2.0).clamp(0.0, 1.0)


def count_params(model):
    return sum(p.numel() for p in model.parameters())

Overwriting src/utils.py


In [10]:
%%writefile src/data.py
"""Pipeline data Fashion-MNIST: gabung 70.000 citra lalu split 80/10/10 dengan seed tetap."""
import os

import torch
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets

from . import config as C

_CACHE = {}


def load_fashion_mnist():
    """Unduh (legal, via torchvision) dan kembalikan tensor x in [0,1] (N,1,28,28) dan y (N,)."""
    train = datasets.FashionMNIST(C.DATA_DIR, train=True, download=True)
    test = datasets.FashionMNIST(C.DATA_DIR, train=False, download=True)
    x = torch.cat([train.data, test.data]).float().div(255.0).unsqueeze(1)
    y = torch.cat([train.targets, test.targets]).long()
    return x, y


def get_splits():
    """Split 80% train, 10% val, 10% test (56.000 / 7.000 / 7.000)."""
    if "splits" in _CACHE:
        return _CACHE["splits"]
    x, y = load_fashion_mnist()
    n = len(x)
    n_train = int(C.SPLIT[0] * n)
    n_val = int(C.SPLIT[1] * n)
    g = torch.Generator().manual_seed(C.SEED)
    perm = torch.randperm(n, generator=g)
    idx = {"train": perm[:n_train], "val": perm[n_train:n_train + n_val], "test": perm[n_train + n_val:]}
    os.makedirs(C.OUT_DIR, exist_ok=True)
    torch.save(idx, os.path.join(C.OUT_DIR, "split_indices.pt"))  # bukti split reprodusibel
    splits = {k: TensorDataset(x[v], y[v]) for k, v in idx.items()}
    _CACHE["splits"] = splits
    return splits


def get_loaders(batch_size=None):
    """shuffle=True hanya untuk train; val/test shuffle=False (sesuai aturan tugas)."""
    bs = batch_size or C.BATCH_SIZE
    s = get_splits()
    g = torch.Generator().manual_seed(C.SEED)
    pin = torch.cuda.is_available()
    loaders = {
        "train": DataLoader(s["train"], batch_size=bs, shuffle=True, generator=g, pin_memory=pin),
        "val": DataLoader(s["val"], batch_size=bs, shuffle=False, pin_memory=pin),
        "test": DataLoader(s["test"], batch_size=bs, shuffle=False, pin_memory=pin),
    }
    print(f"Split -> train: {len(s['train'])} | val: {len(s['val'])} | test: {len(s['test'])} "
          f"| batch_size: {bs}")
    return loaders


def get_test_tensors():
    s = get_splits()["test"]
    return s.tensors[0], s.tensors[1]


def pick_one_per_class(n_classes=10):
    """Ambil citra uji pertama untuk setiap kelas (untuk figur yang konsisten)."""
    x, y = get_test_tensors()
    idx = [int((y == c).nonzero()[0]) for c in range(n_classes)]
    return x[idx], y[idx]

Overwriting src/data.py


In [11]:
%%writefile src/models/__init__.py
from .mlp import MLPClassifier, MLPDecoder
from .autoencoders import Autoencoder, VAE, vae_loss
from .gans import MLPGenerator, MLPDiscriminator, DCGANGenerator, DCGANDiscriminator, dcgan_weights_init
from .ddpm import UNet, Diffusion, EMA

Overwriting src/models/__init__.py


In [12]:
%%writefile src/models/mlp.py
"""MLP Classifier (784 -> 256 -> 64 -> 10) dan Decoder MLP terpisah (64 -> 256 -> 784)."""
import torch.nn as nn
import torch.nn.functional as F

from .. import config as C


class MLPClassifier(nn.Module):
    """Classifier diskriminatif P(y|x). fc1=W1, fc2=W2, fc3=W3."""

    def __init__(self, in_dim=C.IMG_DIM, hidden=C.HIDDEN_DIM, latent=C.LATENT_DIM, n_classes=10, p_drop=0.2):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)      # W1: 256 x 784
        self.fc2 = nn.Linear(hidden, latent)      # W2: 64 x 256
        self.fc3 = nn.Linear(latent, n_classes)   # W3: 10 x 64
        self.drop = nn.Dropout(p_drop)

    def features(self, x):
        """z_MLP in R^64: representasi laten sebelum lapisan klasifikasi akhir."""
        h = self.drop(F.relu(self.fc1(x.flatten(1))))
        return F.relu(self.fc2(h))

    def forward(self, x):
        return self.fc3(self.drop(self.features(x)))


class MLPDecoder(nn.Module):
    """Decoder 64 -> 256 -> 784 (dipakai untuk Frozen Latent Decoding, Tahap 4)."""

    def __init__(self, latent=C.LATENT_DIM, hidden=C.HIDDEN_DIM, out_dim=C.IMG_DIM):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(), nn.Linear(hidden, out_dim), nn.Sigmoid())

    def forward(self, z):
        return self.net(z).view(-1, *C.IMG_SHAPE)

Overwriting src/models/mlp.py


In [13]:
%%writefile src/models/autoencoders.py
"""Vanilla / Sparse Autoencoder (arsitektur sama, beda loss) dan Variational Autoencoder."""
import torch
import torch.nn as nn
import torch.nn.functional as F

from .. import config as C


class Autoencoder(nn.Module):
    """Encoder 784 -> 256 (ReLU) -> 64 (linear), Decoder 64 -> 256 (ReLU) -> 784 (Sigmoid).

    Vanilla AE : L = MSE(x, x_hat)
    Sparse AE  : L = MSE(x, x_hat) + lambda * ||h||_1   (h = aktivasi hidden 256 dimensi)
    """

    def __init__(self, in_dim=C.IMG_DIM, hidden=C.HIDDEN_DIM, latent=C.LATENT_DIM):
        super().__init__()
        self.enc1 = nn.Linear(in_dim, hidden)
        self.enc2 = nn.Linear(hidden, latent)
        self.dec1 = nn.Linear(latent, hidden)
        self.dec2 = nn.Linear(hidden, in_dim)

    def encode(self, x):
        h = F.relu(self.enc1(x.flatten(1)))
        z = self.enc2(h)
        return z, h

    def decode(self, z):
        return torch.sigmoid(self.dec2(F.relu(self.dec1(z)))).view(-1, *C.IMG_SHAPE)

    def forward(self, x):
        z, h = self.encode(x)
        return self.decode(z), z, h


class VAE(nn.Module):
    """q(z|x) = N(mu, diag(sigma^2)), p(z) = N(0, I). Arsitektur sepadan dengan AE."""

    def __init__(self, in_dim=C.IMG_DIM, hidden=C.HIDDEN_DIM, latent=C.LATENT_DIM):
        super().__init__()
        self.latent = latent
        self.enc1 = nn.Linear(in_dim, hidden)
        self.fc_mu = nn.Linear(hidden, latent)
        self.fc_logvar = nn.Linear(hidden, latent)
        self.dec1 = nn.Linear(latent, hidden)
        self.dec2 = nn.Linear(hidden, in_dim)

    def encode(self, x):
        h = F.relu(self.enc1(x.flatten(1)))
        return self.fc_mu(h), self.fc_logvar(h)

    @staticmethod
    def reparameterize(mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def decode(self, z):
        return torch.sigmoid(self.dec2(F.relu(self.dec1(z)))).view(-1, *C.IMG_SHAPE)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def vae_loss(x_hat, x, mu, logvar):
    """Negative ELBO = BCE rekonstruksi + KL(q(z|x) || N(0,I)), dirata-rata per citra."""
    b = x.size(0)
    bce = F.binary_cross_entropy(x_hat, x, reduction="sum") / b
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / b
    return bce + kld, bce, kld

Overwriting src/models/autoencoders.py


In [14]:
%%writefile src/models/gans.py
"""Vanilla GAN (fully connected) dan DCGAN (konvolusional penuh) untuk citra 28x28."""
import torch.nn as nn

from .. import config as C


class MLPGenerator(nn.Module):
    """G: z (100) -> 256 -> 512 -> 1024 -> 784 (Tanh)."""

    def __init__(self, z_dim=C.NOISE_DIM, out_dim=C.IMG_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 1024), nn.LeakyReLU(0.2),
            nn.Linear(1024, out_dim), nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z).view(-1, *C.IMG_SHAPE)


class MLPDiscriminator(nn.Module):
    """D: 784 -> 512 -> 256 -> 1 (logit)."""

    def __init__(self, in_dim=C.IMG_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        return self.net(x.flatten(1)).view(-1)


class DCGANGenerator(nn.Module):
    """Prinsip DCGAN: ConvTranspose (tanpa pooling), BatchNorm, ReLU, output Tanh.
    z (100x1x1) -> 256x7x7 -> 128x14x14 -> 1x28x28
    """

    def __init__(self, z_dim=C.NOISE_DIM, ngf=128):
        super().__init__()
        self.z_dim = z_dim
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, ngf * 2, 7, 1, 0, bias=False), nn.BatchNorm2d(ngf * 2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False), nn.BatchNorm2d(ngf), nn.ReLU(True),
            nn.ConvTranspose2d(ngf, 1, 4, 2, 1), nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z.view(-1, self.z_dim, 1, 1))


class DCGANDiscriminator(nn.Module):
    """Strided conv (tanpa pooling), BatchNorm, LeakyReLU(0.2). 1x28x28 -> 64x14x14 -> 128x7x7 -> 1."""

    def __init__(self, ndf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, ndf, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False), nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf * 2, 1, 7, 1, 0),
        )

    def forward(self, x):
        return self.net(x).view(-1)


def dcgan_weights_init(m):
    """Inisialisasi bobot N(0, 0.02) sesuai paper DCGAN (Radford et al., 2016)."""
    name = m.__class__.__name__
    if "Conv" in name:
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif "BatchNorm" in name:
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

Overwriting src/models/gans.py


In [15]:
%%writefile src/models/ddpm.py
"""DDPM (Ho et al., 2020): U-Net prediktor derau + jadwal difusi linear + EMA."""
import copy
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from .. import config as C


def timestep_embedding(t, dim):
    """Sinusoidal positional embedding untuk timestep t."""
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    args = t.float()[:, None] * freqs[None]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=1)


class ResBlock(nn.Module):
    def __init__(self, cin, cout, tdim, groups=8, p_drop=0.1):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, cin)
        self.conv1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.temb = nn.Linear(tdim, cout)
        self.norm2 = nn.GroupNorm(groups, cout)
        self.drop = nn.Dropout(p_drop)
        self.conv2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x, temb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.temb(F.silu(temb))[:, :, None, None]
        h = self.conv2(self.drop(F.silu(self.norm2(h))))
        return h + self.skip(x)


class UNet(nn.Module):
    """U-Net kecil untuk 28x28: 28 -> 14 -> 7 -> 14 -> 28 dengan skip connection."""

    def __init__(self, ch=C.DDPM_BASE_CH, tdim=256):
        super().__init__()
        self.ch = ch
        self.time_mlp = nn.Sequential(nn.Linear(ch, tdim), nn.SiLU(), nn.Linear(tdim, tdim))
        self.conv_in = nn.Conv2d(1, ch, 3, padding=1)
        self.d1a, self.d1b = ResBlock(ch, ch, tdim), ResBlock(ch, ch, tdim)
        self.down1 = nn.Conv2d(ch, ch, 3, stride=2, padding=1)                  # 28 -> 14
        self.d2a, self.d2b = ResBlock(ch, 2 * ch, tdim), ResBlock(2 * ch, 2 * ch, tdim)
        self.down2 = nn.Conv2d(2 * ch, 2 * ch, 3, stride=2, padding=1)          # 14 -> 7
        self.m1, self.m2 = ResBlock(2 * ch, 2 * ch, tdim), ResBlock(2 * ch, 2 * ch, tdim)
        self.up2 = nn.Sequential(nn.Upsample(scale_factor=2, mode="nearest"), nn.Conv2d(2 * ch, 2 * ch, 3, padding=1))
        self.u2a, self.u2b = ResBlock(4 * ch, 2 * ch, tdim), ResBlock(2 * ch, 2 * ch, tdim)
        self.up1 = nn.Sequential(nn.Upsample(scale_factor=2, mode="nearest"), nn.Conv2d(2 * ch, 2 * ch, 3, padding=1))
        self.u1a, self.u1b = ResBlock(3 * ch, ch, tdim), ResBlock(ch, ch, tdim)
        self.norm_out = nn.GroupNorm(8, ch)
        self.conv_out = nn.Conv2d(ch, 1, 3, padding=1)

    def forward(self, x, t):
        temb = self.time_mlp(timestep_embedding(t, self.ch))
        x = self.conv_in(x)
        s1 = self.d1b(self.d1a(x, temb), temb)
        x = self.down1(s1)
        s2 = self.d2b(self.d2a(x, temb), temb)
        x = self.down2(s2)
        x = self.m2(self.m1(x, temb), temb)
        x = self.up2(x)
        x = self.u2b(self.u2a(torch.cat([x, s2], 1), temb), temb)
        x = self.up1(x)
        x = self.u1b(self.u1a(torch.cat([x, s1], 1), temb), temb)
        return self.conv_out(F.silu(self.norm_out(x)))


class Diffusion:
    """Forward q(x_t|x_0) dan reverse p(x_{t-1}|x_t) dengan jadwal beta linear."""

    def __init__(self, T=C.DDPM_T, beta_start=C.DDPM_BETA_START, beta_end=C.DDPM_BETA_END, device="cpu"):
        self.T = T
        self.device = device
        self.betas = torch.linspace(beta_start, beta_end, T, device=device)
        self.alphas = 1.0 - self.betas
        self.alpha_bar = torch.cumprod(self.alphas, dim=0)
        self.sqrt_ab = self.alpha_bar.sqrt()
        self.sqrt_1m_ab = (1.0 - self.alpha_bar).sqrt()
        self.sqrt_recip_a = 1.0 / self.alphas.sqrt()
        self.eps_coef = self.betas / self.sqrt_1m_ab
        self.sigma = self.betas.sqrt()

    def q_sample(self, x0, t, noise):
        """x_t = sqrt(abar_t) x_0 + sqrt(1 - abar_t) eps."""
        return self.sqrt_ab[t].view(-1, 1, 1, 1) * x0 + self.sqrt_1m_ab[t].view(-1, 1, 1, 1) * noise

    def loss(self, model, x0):
        """L_simple = E || eps - eps_theta(x_t, t) ||^2 (supervised noise prediction)."""
        t = torch.randint(0, self.T, (x0.size(0),), device=x0.device)
        noise = torch.randn_like(x0)
        return F.mse_loss(model(self.q_sample(x0, t, noise), t), noise)

    @torch.no_grad()
    def sample(self, model, n, use_amp=False, traj_steps=None, generator=None):
        """Reverse denoising x_T ~ N(0,I) -> x_0 (T langkah). Output di rentang [-1, 1]."""
        model.eval()
        amp = use_amp and str(self.device).startswith("cuda")
        x = torch.randn(n, *C.IMG_SHAPE, device=self.device, generator=generator)
        traj = {}
        if traj_steps is not None and self.T in traj_steps:
            traj[self.T] = x.clamp(-1, 1).cpu()
        for i in reversed(range(self.T)):
            t = torch.full((n,), i, device=self.device, dtype=torch.long)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
                eps = model(x, t)
            eps = eps.float()
            mean = self.sqrt_recip_a[i] * (x - self.eps_coef[i] * eps)
            if i > 0:
                z = torch.randn(x.shape, device=self.device, generator=generator)
                x = mean + self.sigma[i] * z
            else:
                x = mean
            if traj_steps is not None and i in traj_steps:
                traj[i] = x.clamp(-1, 1).cpu()
        x = x.clamp(-1, 1)
        return (x, traj) if traj_steps is not None else x


class EMA:
    """Exponential Moving Average bobot model (meningkatkan kualitas sampel DDPM)."""

    def __init__(self, model, decay=C.EMA_DECAY):
        self.decay = decay
        self.shadow = copy.deepcopy(model).eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for ps, p in zip(self.shadow.parameters(), model.parameters()):
            ps.mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)
        for bs, b in zip(self.shadow.buffers(), model.buffers()):
            bs.copy_(b)

Overwriting src/models/ddpm.py


In [16]:
%%writefile src/train.py
"""Training loop untuk 7 model. Setiap epoch disimpan ke checkpoint sehingga bisa di-resume.

Bila checkpoint sudah mencapai jumlah epoch target, training dilewati (tidak mengulang dari nol).
"""
import os

import torch
import torch.nn.functional as F
from torchvision.utils import save_image

from . import config as C
from .models import (MLPClassifier, MLPDecoder, Autoencoder, VAE, vae_loss, MLPGenerator,
                     MLPDiscriminator, DCGANGenerator, DCGANDiscriminator, dcgan_weights_init,
                     UNet, Diffusion, EMA)
from .utils import Timer, set_seed, save_ckpt, load_ckpt, save_json, to_pm1, to_01, count_params


# ------------------------------------------------------------------ helper resume / log
def _resume(name, modules, optims, device):
    ck = load_ckpt(name, device)
    if ck is None:
        return 0, {}, {}
    for k, m in modules.items():
        m.load_state_dict(ck["models"][k])
    for k, o in optims.items():
        o.load_state_dict(ck["optims"][k])
    print(f"[{name}] checkpoint ditemukan -> lanjut dari epoch {ck['epoch']}")
    return ck["epoch"], ck["logs"], ck.get("extra", {})


def _save(name, epoch, modules, optims, logs, extra=None):
    save_ckpt(name, {
        "epoch": epoch,
        "models": {k: m.state_dict() for k, m in modules.items()},
        "optims": {k: o.state_dict() for k, o in optims.items()},
        "logs": logs,
        "extra": extra or {},
    })
    save_json(logs, os.path.join(C.LOG_DIR, f"{name}.json"))


def _log(logs, **kv):
    for k, v in kv.items():
        logs.setdefault(k, []).append(float(v))


def _header(name, model, epochs, start):
    print(f"\n===== {name} | params: {count_params(model):,} | epoch target: {epochs} | mulai: {start} =====")
    if start >= epochs:
        print(f"[{name}] sudah selesai dilatih -> dilewati.")


@torch.no_grad()
def accuracy(model, loader, device):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += y.numel()
    return correct / total


# ------------------------------------------------------------------ 1. MLP Classifier
def train_mlp_classifier(loaders, device):
    name = "mlp"
    set_seed()
    model = MLPClassifier().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=C.LR[name])
    start, logs, extra = _resume(name, {"model": model}, {"opt": opt}, device)
    best_state, best_val = extra.get("best_state"), extra.get("best_val", -1.0)
    E = C.EPOCHS[name]
    _header(name, model, E, start)
    for ep in range(start, E):
        model.train()
        tot = n = 0
        with Timer(device) as tm:
            for x, y in loaders["train"]:
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                loss = F.cross_entropy(model(x), y)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
                tot += loss.item() * x.size(0)
                n += x.size(0)
        val_acc = accuracy(model, loaders["val"], device)
        _log(logs, epoch_time=tm.elapsed, train_loss=tot / n, val_acc=val_acc)
        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        _save(name, ep + 1, {"model": model}, {"opt": opt}, logs, {"best_state": best_state, "best_val": best_val})
        print(f"[{name}] epoch {ep + 1:>3}/{E} | CE loss {tot / n:.4f} | val acc {val_acc:.4f} | {tm.elapsed:.1f}s")
    model.load_state_dict(best_state)
    model.eval()
    test_acc = accuracy(model, loaders["test"], device)
    status = "MEMENUHI" if test_acc >= C.MLP_MIN_ACC else "BELUM MEMENUHI (tambah epoch!)"
    print(f"[{name}] Test accuracy (bobot val terbaik): {test_acc:.4f} -> syarat >= {C.MLP_MIN_ACC:.0%}: {status}")
    logs["test_acc"] = test_acc
    save_json(logs, os.path.join(C.LOG_DIR, f"{name}.json"))
    return model, logs


# ------------------------------------------------------------------ 2 & 3. Vanilla AE / Sparse AE
def train_autoencoder(loaders, device, sparse=False):
    name = "sparse_ae" if sparse else "vanilla_ae"
    lam = C.SPARSE_LAMBDA if sparse else 0.0
    set_seed()
    model = Autoencoder().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=C.LR[name])
    start, logs, _ = _resume(name, {"model": model}, {"opt": opt}, device)
    E = C.EPOCHS[name]
    _header(name, model, E, start)
    for ep in range(start, E):
        model.train()
        s_loss = s_mse = s_l1 = n = 0
        with Timer(device) as tm:
            for x, _ in loaders["train"]:
                x = x.to(device, non_blocking=True)
                x_hat, z, h = model(x)
                mse = F.mse_loss(x_hat, x)
                l1 = h.abs().sum(1).mean()            # ||h||_1 rata-rata per citra
                loss = mse + lam * l1
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
                b = x.size(0)
                s_loss += loss.item() * b; s_mse += mse.item() * b; s_l1 += l1.item() * b; n += b
        val_mse = _val_recon(lambda v: model(v)[0], loaders["val"], device, model)
        _log(logs, epoch_time=tm.elapsed, train_loss=s_loss / n, train_mse=s_mse / n,
             train_l1=s_l1 / n, val_mse=val_mse)
        _save(name, ep + 1, {"model": model}, {"opt": opt}, logs)
        print(f"[{name}] epoch {ep + 1:>3}/{E} | loss {s_loss / n:.5f} | MSE {s_mse / n:.5f} | "
              f"L1(h) {s_l1 / n:.2f} | val MSE {val_mse:.5f} | {tm.elapsed:.1f}s")
    model.eval()
    return model, logs


@torch.no_grad()
def _val_recon(fn, loader, device, model):
    model.eval()
    tot = n = 0
    for x, _ in loader:
        x = x.to(device)
        tot += F.mse_loss(fn(x), x, reduction="sum").item() / x[0].numel()
        n += x.size(0)
    return tot / n


# ------------------------------------------------------------------ 4. VAE
def train_vae(loaders, device):
    name = "vae"
    set_seed()
    model = VAE().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=C.LR[name])
    start, logs, _ = _resume(name, {"model": model}, {"opt": opt}, device)
    E = C.EPOCHS[name]
    _header(name, model, E, start)
    for ep in range(start, E):
        model.train()
        s_loss = s_bce = s_kld = n = 0
        with Timer(device) as tm:
            for x, _ in loaders["train"]:
                x = x.to(device, non_blocking=True)
                x_hat, mu, logvar = model(x)
                loss, bce, kld = vae_loss(x_hat, x, mu, logvar)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
                b = x.size(0)
                s_loss += loss.item() * b; s_bce += bce.item() * b; s_kld += kld.item() * b; n += b
        # validasi: loss ELBO + MSE rekonstruksi dari mu (deterministik)
        model.eval()
        v_loss = v_n = 0
        with torch.no_grad():
            for x, _ in loaders["val"]:
                x = x.to(device)
                xh, mu, lv = model(x)
                v_loss += vae_loss(xh, x, mu, lv)[0].item() * x.size(0)
                v_n += x.size(0)
        val_mse = _val_recon(lambda v: model.decode(model.encode(v)[0]), loaders["val"], device, model)
        _log(logs, epoch_time=tm.elapsed, train_loss=s_loss / n, train_bce=s_bce / n, train_kld=s_kld / n,
             val_loss=v_loss / v_n, val_mse=val_mse)
        _save(name, ep + 1, {"model": model}, {"opt": opt}, logs)
        print(f"[{name}] epoch {ep + 1:>3}/{E} | -ELBO {s_loss / n:.2f} (BCE {s_bce / n:.2f} + KL {s_kld / n:.2f}) "
              f"| val -ELBO {v_loss / v_n:.2f} | val MSE {val_mse:.5f} | {tm.elapsed:.1f}s")
    model.eval()
    return model, logs


# ------------------------------------------------------------------ 5. Frozen Latent Decoder (Tahap 4)
def train_frozen_decoder(clf, loaders, device):
    name = "frozen_decoder"
    set_seed()
    clf.eval()
    for p in clf.parameters():
        p.requires_grad = False                     # seluruh bobot MLP Classifier dibekukan
    dec = MLPDecoder().to(device)
    opt = torch.optim.Adam(dec.parameters(), lr=C.LR[name])
    start, logs, _ = _resume(name, {"model": dec}, {"opt": opt}, device)
    E = C.EPOCHS[name]
    _header(name, dec, E, start)
    for ep in range(start, E):
        dec.train()
        tot = n = 0
        with Timer(device) as tm:
            for x, _ in loaders["train"]:
                x = x.to(device, non_blocking=True)
                with torch.no_grad():
                    z = clf.features(x)             # z_MLP beku
                loss = F.mse_loss(dec(z), x)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
                tot += loss.item() * x.size(0); n += x.size(0)
        val_mse = _val_recon(lambda v: dec(clf.features(v)), loaders["val"], device, dec)
        _log(logs, epoch_time=tm.elapsed, train_loss=tot / n, val_mse=val_mse)
        _save(name, ep + 1, {"model": dec}, {"opt": opt}, logs)
        print(f"[{name}] epoch {ep + 1:>3}/{E} | MSE {tot / n:.5f} | val MSE {val_mse:.5f} | {tm.elapsed:.1f}s")
    dec.eval()
    return dec, logs


# ------------------------------------------------------------------ 6 & 7. Vanilla GAN / DCGAN
def train_gan(loaders, device, kind="vanilla_gan", snapshot_every=5):
    assert kind in ("vanilla_gan", "dcgan")
    name = kind
    set_seed()
    if kind == "vanilla_gan":
        G, D = MLPGenerator().to(device), MLPDiscriminator().to(device)
    else:
        G, D = DCGANGenerator().to(device), DCGANDiscriminator().to(device)
        G.apply(dcgan_weights_init)
        D.apply(dcgan_weights_init)
    optG = torch.optim.Adam(G.parameters(), lr=C.LR[name], betas=C.GAN_BETAS)
    optD = torch.optim.Adam(D.parameters(), lr=C.LR[name], betas=C.GAN_BETAS)
    start, logs, _ = _resume(name, {"G": G, "D": D}, {"optG": optG, "optD": optD}, device)
    E = C.EPOCHS[name]
    print(f"\n[{name}] params D: {count_params(D):,}")
    _header(name, G, E, start)
    g_fixed = torch.Generator(device=device).manual_seed(C.SEED)
    fixed_z = torch.randn(64, C.NOISE_DIM, device=device, generator=g_fixed)
    snap_dir = os.path.join(C.FIG_DIR, "training_progress")
    os.makedirs(snap_dir, exist_ok=True)
    bce = torch.nn.BCEWithLogitsLoss()
    for ep in range(start, E):
        G.train(); D.train()
        sD = sG = sDx = sDgz = n = 0
        itD, itG = [], []
        with Timer(device) as tm:
            for x, _ in loaders["train"]:
                real = to_pm1(x.to(device, non_blocking=True))
                b = real.size(0)
                ones, zeros = torch.ones(b, device=device), torch.zeros(b, device=device)
                fake = G(torch.randn(b, C.NOISE_DIM, device=device))
                # --- Discriminator: max log D(x) + log(1 - D(G(z)))
                d_real, d_fake = D(real), D(fake.detach())
                lossD = bce(d_real, ones) + bce(d_fake, zeros)
                optD.zero_grad(set_to_none=True)
                lossD.backward()
                optD.step()
                # --- Generator (non-saturating): max log D(G(z))
                lossG = bce(D(fake), ones)
                optG.zero_grad(set_to_none=True)
                lossG.backward()
                optG.step()
                itD.append(lossD.item()); itG.append(lossG.item())
                sD += lossD.item() * b; sG += lossG.item() * b
                sDx += torch.sigmoid(d_real).mean().item() * b
                sDgz += torch.sigmoid(d_fake).mean().item() * b
                n += b
        logs.setdefault("iter_lossD", []).extend(itD)
        logs.setdefault("iter_lossG", []).extend(itG)
        _log(logs, epoch_time=tm.elapsed, lossD=sD / n, lossG=sG / n, D_x=sDx / n, D_G_z=sDgz / n)
        _save(name, ep + 1, {"G": G, "D": D}, {"optG": optG, "optD": optD}, logs)
        if (ep + 1) % snapshot_every == 0 or ep + 1 == E:
            G.eval()
            with torch.no_grad():
                save_image(to_01(G(fixed_z)), os.path.join(snap_dir, f"{name}_epoch{ep + 1:03d}.png"), nrow=8)
        print(f"[{name}] epoch {ep + 1:>3}/{E} | loss D {sD / n:.4f} | loss G {sG / n:.4f} | "
              f"D(x) {sDx / n:.3f} | D(G(z)) {sDgz / n:.3f} | {tm.elapsed:.1f}s")
    G.eval()
    return G, logs


# ------------------------------------------------------------------ 8. DDPM
def train_ddpm(loaders, device):
    name = "ddpm"
    set_seed()
    model = UNet().to(device)
    ema = EMA(model)
    diff = Diffusion(device=device)
    opt = torch.optim.Adam(model.parameters(), lr=C.LR[name])
    use_amp = C.USE_AMP and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda") if use_amp else None
    start, logs, extra = _resume(name, {"model": model, "ema": ema.shadow}, {"opt": opt}, device)
    if scaler is not None and extra.get("scaler"):
        scaler.load_state_dict(extra["scaler"])
    E = C.EPOCHS[name]
    _header(name, model, E, start)
    print(f"[{name}] T = {diff.T} | mixed precision (AMP): {use_amp}")
    for ep in range(start, E):
        model.train()
        tot = n = 0
        with Timer(device) as tm:
            for x, _ in loaders["train"]:
                x0 = to_pm1(x.to(device, non_blocking=True))
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                    loss = diff.loss(model, x0)
                opt.zero_grad(set_to_none=True)
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
                ema.update(model)
                if not torch.isfinite(loss):
                    raise RuntimeError("Loss DDPM NaN/Inf - hentikan dan periksa learning rate.")
                tot += loss.item() * x0.size(0); n += x0.size(0)
        model.eval()
        v_tot = v_n = 0
        with torch.no_grad():
            for x, _ in loaders["val"]:
                x0 = to_pm1(x.to(device))
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                    v_tot += diff.loss(model, x0).item() * x0.size(0)
                v_n += x0.size(0)
        _log(logs, epoch_time=tm.elapsed, train_loss=tot / n, val_loss=v_tot / v_n)
        _save(name, ep + 1, {"model": model, "ema": ema.shadow}, {"opt": opt}, logs,
              {"scaler": scaler.state_dict() if scaler is not None else None})
        sisa = (E - ep - 1) * tm.elapsed / 60
        print(f"[{name}] epoch {ep + 1:>3}/{E} | noise MSE {tot / n:.5f} | val {v_tot / v_n:.5f} | "
              f"{tm.elapsed:.1f}s | perkiraan sisa {sisa:.1f} menit")
    return ema.shadow, logs


@torch.no_grad()
def generate_ddpm_samples(device, n=None, chunk=500):
    """Sampling DDPM (model EMA) untuk FID + trajektori denoising. Disimpan per-chunk (bisa resume)."""
    n = n or C.N_FID
    out_path = os.path.join(C.SAMPLE_DIR, "ddpm_fid_samples.pt")
    traj_path = os.path.join(C.SAMPLE_DIR, "ddpm_trajectory.pt")
    os.makedirs(C.SAMPLE_DIR, exist_ok=True)
    model, _ = load_model("ddpm", device)
    diff = Diffusion(device=device)
    use_amp = C.USE_AMP and device.type == "cuda"
    if not os.path.exists(traj_path):
        g = torch.Generator(device=device).manual_seed(C.SEED)
        _, traj = diff.sample(model, 8, use_amp=use_amp, traj_steps=C.DDPM_TRAJ_STEPS, generator=g)
        torch.save(traj, traj_path)
        print("[ddpm] trajektori denoising tersimpan.")
    if os.path.exists(out_path):
        print(f"[ddpm] sampel FID sudah ada: {out_path}")
        return torch.load(out_path).float() / 255.0
    parts = []
    n_chunks = (n + chunk - 1) // chunk
    for i in range(n_chunks):
        p = os.path.join(C.SAMPLE_DIR, f"ddpm_part{i:02d}.pt")
        if not os.path.exists(p):
            b = min(chunk, n - i * chunk)
            g = torch.Generator(device=device).manual_seed(C.SEED + 1000 + i)
            with Timer(device) as tm:
                x = diff.sample(model, b, use_amp=use_amp, generator=g)
            torch.save((to_01(x) * 255).round().to(torch.uint8).cpu(), p)
            print(f"[ddpm] chunk {i + 1}/{n_chunks} ({b} sampel) selesai dalam {tm.elapsed:.1f}s")
        parts.append(torch.load(p))
    samples = torch.cat(parts)[:n]
    torch.save(samples, out_path)
    for i in range(n_chunks):
        os.remove(os.path.join(C.SAMPLE_DIR, f"ddpm_part{i:02d}.pt"))
    print(f"[ddpm] {len(samples)} sampel FID tersimpan -> {out_path}")
    return samples.float() / 255.0


# ------------------------------------------------------------------ memuat model terlatih
def load_model(name, device):
    ck = load_ckpt(name, device)
    if ck is None:
        raise FileNotFoundError(f"Checkpoint '{name}' belum ada. Jalankan training '{name}' dulu.")
    if ck["epoch"] < C.EPOCHS[name]:
        print(f"PERINGATAN: '{name}' baru {ck['epoch']}/{C.EPOCHS[name]} epoch.")
    if name == "mlp":
        model = MLPClassifier()
        state = ck["extra"]["best_state"]
    elif name == "frozen_decoder":
        model, state = MLPDecoder(), ck["models"]["model"]
    elif name in ("vanilla_ae", "sparse_ae"):
        model, state = Autoencoder(), ck["models"]["model"]
    elif name == "vae":
        model, state = VAE(), ck["models"]["model"]
    elif name == "vanilla_gan":
        model, state = MLPGenerator(), ck["models"]["G"]
    elif name == "dcgan":
        model, state = DCGANGenerator(), ck["models"]["G"]
    elif name == "ddpm":
        model, state = UNet(), ck["models"]["ema"]
    else:
        raise ValueError(name)
    model.load_state_dict(state)
    model.to(device).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model, ck["logs"]

Overwriting src/train.py


In [17]:
%%writefile src/experiments.py
"""Eksperimen Bagian 2 (batasan MLP Classifier) dan eksperimen interpolasi laten (Q4 / Figur 4)."""
import torch
import torch.nn.functional as F

from . import config as C
from .utils import to_01


# ------------------------------------------------------------------ Tahap 2: proyeksi balik dari one-hot y
@torch.no_grad()
def topdown_from_class(clf):
    """z_hat = ReLU(W3^T y), h_hat = ReLU(W2^T z_hat), x_hat = Sigmoid(W1^T h_hat), tanpa pelatihan tambahan."""
    W1, W2, W3 = clf.fc1.weight, clf.fc2.weight, clf.fc3.weight     # (256,784), (64,256), (10,64)
    y = torch.eye(10, device=W1.device)                               # 10 vektor one-hot (baris)
    z_hat = F.relu(y @ W3)                                            # = (W3^T y)^T  -> (10, 64)
    h_hat = F.relu(z_hat @ W2)                                        # = (W2^T z)^T  -> (10, 256)
    x_hat = torch.sigmoid(h_hat @ W1)                                 # = (W1^T h)^T  -> (10, 784)
    return x_hat.view(-1, *C.IMG_SHAPE)


# ------------------------------------------------------------------ Tahap 3: proyeksi balik dari z_MLP
@torch.no_grad()
def latent_transpose(clf, z):
    """x_hat = Sigmoid(W1^T ReLU(W2^T z_MLP))."""
    W1, W2 = clf.fc1.weight, clf.fc2.weight
    return torch.sigmoid(F.relu(z @ W2) @ W1).view(-1, *C.IMG_SHAPE)


@torch.no_grad()
def orthogonality_check(clf):
    """Bukti numerik W^T W != I: ||W^T W - I||_F dan error proyeksi balik W^T W x vs x."""
    out = {}
    for name, layer in [("W1", clf.fc1), ("W2", clf.fc2), ("W3", clf.fc3)]:
        W = layer.weight
        WtW = W.t() @ W
        I = torch.eye(WtW.size(0), device=W.device)
        out[name] = {
            "shape": list(W.shape),
            "rank": int(torch.linalg.matrix_rank(W).item()),
            "fro_norm_WtW_minus_I": float(torch.linalg.norm(WtW - I).item()),
            "rel_fro_norm": float((torch.linalg.norm(WtW - I) / torch.linalg.norm(I)).item()),
        }
    return out


# ------------------------------------------------------------------ Tahap 5: statistik ruang laten MLP
@torch.no_grad()
def latent_stats(clf, loader, device):
    zs = []
    for x, _ in loader:
        zs.append(clf.features(x.to(device)))
    z = torch.cat(zs)
    return z.mean(0), z.std(0), z


@torch.no_grad()
def mlp_prior_sampling(decoder, mu, sigma, n, device, seed=C.SEED):
    """z_rand ~ N(mu_z, diag(sigma_z^2)) -> Decoder Tahap 4."""
    g = torch.Generator(device=device).manual_seed(seed)
    z = mu + sigma * torch.randn(n, mu.numel(), device=device, generator=g)
    return decoder(z)


# ------------------------------------------------------------------ Interpolasi laten (Figur 4)
def _lerp(za, zb, alphas):
    return torch.cat([(1 - a) * za + a * zb for a in alphas])


@torch.no_grad()
def interpolate_autoencoder(model, xa, xb, alphas=C.INTERP_ALPHAS):
    """Vanilla AE / Sparse AE: z = encoder(x). VAE: z = mu(x)."""
    za, zb = model.encode(xa)[0], model.encode(xb)[0]
    return model.decode(_lerp(za, zb, alphas))


@torch.no_grad()
def find_gan_endpoints(G, clf, device, class_a=C.INTERP_CLASS_A, class_b=C.INTERP_CLASS_B, n=2048, seed=C.SEED):
    """GAN tidak punya encoder -> pilih z_A, z_B (dari prior) yang hasilnya paling yakin
    diklasifikasikan sebagai kelas A dan kelas B oleh MLP Classifier."""
    g = torch.Generator(device=device).manual_seed(seed)
    z = torch.randn(n, C.NOISE_DIM, device=device, generator=g)
    probs = torch.cat([F.softmax(clf(to_01(G(zc))), 1) for zc in z.split(512)])
    ia, ib = probs[:, class_a].argmax(), probs[:, class_b].argmax()
    return z[ia:ia + 1], z[ib:ib + 1], float(probs[ia, class_a]), float(probs[ib, class_b])


@torch.no_grad()
def interpolate_gan(G, za, zb, alphas=C.INTERP_ALPHAS):
    return to_01(G(_lerp(za, zb, alphas)))

Overwriting src/experiments.py


In [18]:
%%writefile src/evaluate.py
"""Evaluasi terstandarisasi pada Test Set: MSE, PSNR, SSIM, FID, latency, training time, stabilitas."""
import os

import numpy as np
import pandas as pd
import torch
from torchmetrics.functional.image import structural_similarity_index_measure as ssim_fn
from torchmetrics.image.fid import FrechetInceptionDistance

from . import config as C
from . import experiments as X
from .data import get_test_tensors
from .models import Diffusion
from .train import load_model, accuracy, generate_ddpm_samples
from .utils import Timer, save_json, to_01, count_params, set_seed


# ------------------------------------------------------------------ metrik rekonstruksi
@torch.no_grad()
def recon_metrics(fn, loader, device):
    """MSE & PSNR dihitung per citra lalu dirata-rata; SSIM (Gaussian 11x11) dirata-rata. Rentang [0,1]."""
    s_mse = s_psnr = s_ssim = n = 0.0
    for x, _ in loader:
        x = x.to(device)
        xh = fn(x).clamp(0, 1)
        mse = ((xh - x) ** 2).flatten(1).mean(1)
        s_mse += mse.sum().item()
        s_psnr += (10 * torch.log10(1.0 / mse.clamp_min(1e-10))).sum().item()
        s_ssim += ssim_fn(xh, x, data_range=1.0).item() * x.size(0)
        n += x.size(0)
    return {"mse": s_mse / n, "psnr": s_psnr / n, "ssim": s_ssim / n}


@torch.no_grad()
def sparsity_stats(model, loader, device):
    """Statistik aktivasi hidden h (256 unit): proporsi unit aktif dan rata-rata ||h||_1."""
    act = l1 = n = 0.0
    for x, _ in loader:
        _, h = model.encode(x.to(device))
        act += (h > 1e-6).float().mean(1).sum().item()
        l1 += h.abs().sum(1).sum().item()
        n += x.size(0)
    return {"active_ratio": act / n, "mean_l1": l1 / n}


# ------------------------------------------------------------------ FID
class FIDEvaluator:
    """FID (Inception-v3, pool 2048). Citra grayscale 28x28 diulang 3 kanal; resize 299 dilakukan internal."""

    def __init__(self, real, device):
        self.device = device
        self.fid = FrechetInceptionDistance(feature=2048, normalize=True, reset_real_features=False).to(device)
        for chunk in real.split(C.FID_BATCH):
            self.fid.update(chunk.repeat(1, 3, 1, 1).to(device), real=True)

    @torch.no_grad()
    def score(self, fake):
        self.fid.reset()                              # hanya statistik fake yang di-reset
        for chunk in fake.split(C.FID_BATCH):
            self.fid.update(chunk.clamp(0, 1).repeat(1, 3, 1, 1).to(self.device), real=False)
        return float(self.fid.compute())


@torch.no_grad()
def generate(kind, model, n, device, seed=C.SEED):
    """Sampling acak dari prior z ~ N(0, I). Output di [0,1], disimpan di CPU."""
    g = torch.Generator(device=device).manual_seed(seed)
    out = []
    for i in range(0, n, 500):
        b = min(500, n - i)
        if kind == "vae":
            out.append(model.decode(torch.randn(b, C.LATENT_DIM, device=device, generator=g)).cpu())
        else:
            out.append(to_01(model(torch.randn(b, C.NOISE_DIM, device=device, generator=g))).cpu())
    return torch.cat(out)


# ------------------------------------------------------------------ latency
@torch.no_grad()
def measure_latency(fn, device, runs, warmup_fn=None, warmup=3):
    """Latency (ms) per batch 64 sampel, dengan warm-up dan cuda.synchronize."""
    for _ in range(warmup):
        (warmup_fn or fn)()
    times = []
    for _ in range(runs):
        with Timer(device) as t:
            fn()
        times.append(t.elapsed * 1000)
    return float(np.mean(times)), float(np.std(times))


def _stability(values, k=5):
    """Koefisien variasi (std/mean) loss pada k epoch terakhir: makin kecil makin stabil."""
    v = np.array(values[-k:], dtype=float)
    return float(np.std(v) / (abs(np.mean(v)) + 1e-12))


def _fmt(v, nd=4):
    if isinstance(v, (float, np.floating)):
        return f"{v:.{nd}f}"
    return str(v)


def df_to_markdown(df):
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(_fmt(row[c]) for c in cols) + " |")
    return "\n".join(lines)


def _save_table(df, stem):
    df.to_csv(os.path.join(C.RESULT_DIR, f"{stem}.csv"), index=False)
    with open(os.path.join(C.RESULT_DIR, f"{stem}.md"), "w") as f:
        f.write(df_to_markdown(df) + "\n")


# ------------------------------------------------------------------ evaluasi utama
def run_evaluation(loaders, device):
    set_seed()
    os.makedirs(C.RESULT_DIR, exist_ok=True)
    names = ["mlp", "frozen_decoder", "vanilla_ae", "sparse_ae", "vae", "vanilla_gan", "dcgan", "ddpm"]
    models, logs = {}, {}
    for n in names:
        models[n], logs[n] = load_model(n, device)
    clf, dec = models["mlp"], models["frozen_decoder"]
    ctx = {"models": models, "logs": logs, "device": device}

    # ---------------- (a) rekonstruksi pada Test Set
    print("Menghitung MSE / PSNR / SSIM pada Test Set ...")
    rec = {
        "vanilla_ae": recon_metrics(lambda x: models["vanilla_ae"](x)[0], loaders["test"], device),
        "sparse_ae": recon_metrics(lambda x: models["sparse_ae"](x)[0], loaders["test"], device),
        "vae": recon_metrics(lambda x: models["vae"].decode(models["vae"].encode(x)[0]), loaders["test"], device),
        "frozen_decoder": recon_metrics(lambda x: dec(clf.features(x)), loaders["test"], device),
        "latent_transpose": recon_metrics(lambda x: X.latent_transpose(clf, clf.features(x)), loaders["test"], device),
    }
    sparsity = {k: sparsity_stats(models[k], loaders["test"], device) for k in ["vanilla_ae", "sparse_ae"]}

    # ---------------- (b) sampel generatif untuk FID
    print(f"Membuat {C.N_FID} sampel per model generatif ...")
    x_test, _ = get_test_tensors()
    g = torch.Generator().manual_seed(C.SEED)
    real = x_test[torch.randperm(len(x_test), generator=g)[:C.N_FID]]
    samples = {
        "vae": generate("vae", models["vae"], C.N_FID, device),
        "vanilla_gan": generate("gan", models["vanilla_gan"], C.N_FID, device),
        "dcgan": generate("gan", models["dcgan"], C.N_FID, device),
        "ddpm": generate_ddpm_samples(device),
    }
    mu_z, sd_z, _ = X.latent_stats(clf, loaders["test"], device)
    samples["mlp_prior"] = X.mlp_prior_sampling(dec, mu_z, sd_z, C.N_FID, device).cpu()
    ctx.update(samples=samples, mu_z=mu_z, sd_z=sd_z)

    print("Menghitung FID (Inception-v3) ...")
    fid_eval = FIDEvaluator(real, device)
    fid = {k: fid_eval.score(v) for k, v in samples.items()}
    for k, v in fid.items():
        print(f"  FID {k:<12}: {v:.2f}")

    # ---------------- (c) latency per batch 64 (fp32, GPU yang sama untuk semua model)
    print("Mengukur inference latency (batch 64) ...")
    B = C.LATENCY_BATCH
    x64 = x_test[:B].to(device)
    zl = torch.randn(B, C.LATENT_DIM, device=device)
    zn = torch.randn(B, C.NOISE_DIM, device=device)
    R = C.LATENCY_RUNS
    lat = {
        "vanilla_ae": measure_latency(lambda: models["vanilla_ae"](x64), device, R),
        "sparse_ae": measure_latency(lambda: models["sparse_ae"](x64), device, R),
        "vae": measure_latency(lambda: models["vae"].decode(zl), device, R),
        "vanilla_gan": measure_latency(lambda: models["vanilla_gan"](zn), device, R),
        "dcgan": measure_latency(lambda: models["dcgan"](zn), device, R),
    }
    diff = Diffusion(device=device)
    t0 = torch.zeros(B, dtype=torch.long, device=device)
    xT = torch.randn(B, *C.IMG_SHAPE, device=device)
    lat["ddpm"] = measure_latency(lambda: diff.sample(models["ddpm"], B, use_amp=False), device,
                                  C.LATENCY_RUNS_DDPM, warmup_fn=lambda: models["ddpm"](xT, t0))
    for k, (m, s) in lat.items():
        print(f"  latency {k:<12}: {m:10.2f} +- {s:.2f} ms / batch {B}")

    # ---------------- (d) training time & stabilitas
    def t_epoch(k):
        return float(np.mean(logs[k]["epoch_time"]))

    main_loss = {"vanilla_ae": "train_mse", "sparse_ae": "train_loss", "vae": "train_loss",
                 "vanilla_gan": "lossG", "dcgan": "lossG", "ddpm": "train_loss"}
    stab = {k: _stability(logs[k][v]) for k, v in main_loss.items()}
    stab_d = {k: _stability(logs[k]["lossD"]) for k in ["vanilla_gan", "dcgan"]}

    # ---------------- Tabel Bagian IV
    rows = [
        ("Vanilla Autoencoder", "vanilla_ae", f"{C.LATENT_DIM}"),
        ("Sparse Autoencoder", "sparse_ae", f"{C.LATENT_DIM} (Bottleneck) / {C.HIDDEN_DIM} (Sparse)"),
        ("Variational Autoencoder (VAE)", "vae", f"{C.LATENT_DIM}"),
        ("Vanilla GAN (MLP)", "vanilla_gan", f"{C.NOISE_DIM} (Prior Noise)"),
        ("DCGAN", "dcgan", f"{C.NOISE_DIM} (Prior Noise)"),
        ("DDPM", "ddpm", "28x28 (Dimensi Citra)"),
    ]
    table = []
    for label, k, zdim in rows:
        r = rec.get(k)
        stab_txt = f"CV loss(G)={stab[k]:.3f}; CV loss(D)={stab_d[k]:.3f}" if k in stab_d else f"CV loss={stab[k]:.4f}"
        table.append({
            "Arsitektur Model": label,
            "Dim Laten": zdim,
            "Training Time / Epoch (s)": round(t_epoch(k), 2),
            "Latency (ms / batch 64)": round(lat[k][0], 2),
            "Test MSE": round(r["mse"], 5) if r else "N/A",
            "Test PSNR (dB)": round(r["psnr"], 2) if r else "N/A",
            "Test SSIM": round(r["ssim"], 4) if r else "N/A",
            "FID Score": round(fid[k], 2) if k in fid else "N/A (Latent Voids)",
            "Stabilitas (5 epoch terakhir)": stab_txt,
        })
    df_main = pd.DataFrame(table)
    _save_table(df_main, "tabel_metrik_bagian_IV")

    # ---------------- Tabel Bagian III (MLP Classifier)
    test_acc = accuracy(clf, loaders["test"], device)
    ortho = X.orthogonality_check(clf)
    df_mlp = pd.DataFrame([
        {"Metode": "Vanilla AE (z_AE, 64-d) [pembanding]", **_rec_cols(rec["vanilla_ae"]), "FID": "N/A"},
        {"Metode": "Frozen Latent Decoder (z_MLP, 64-d)", **_rec_cols(rec["frozen_decoder"]), "FID": "N/A"},
        {"Metode": "Latent Transpose Projection W^T (z_MLP)", **_rec_cols(rec["latent_transpose"]), "FID": "N/A"},
        {"Metode": "Prior sampling z~N(mu_z, diag(sigma_z^2)) -> Decoder", "Test MSE": "N/A",
         "Test PSNR (dB)": "N/A", "Test SSIM": "N/A", "FID": round(fid["mlp_prior"], 2)},
    ])
    _save_table(df_mlp, "tabel_mlp_bagian_III")
    df_ortho = pd.DataFrame([{"Matriks": k, "Shape": "x".join(map(str, v["shape"])), "Rank": v["rank"],
                              "||W^T W - I||_F": round(v["fro_norm_WtW_minus_I"], 3),
                              "Relatif thd ||I||_F": round(v["rel_fro_norm"], 3)} for k, v in ortho.items()])
    _save_table(df_ortho, "tabel_ortogonalitas_bobot")
    df_sp = pd.DataFrame([{"Model": k, "Proporsi unit hidden aktif": round(v["active_ratio"], 4),
                           "Rata-rata ||h||_1": round(v["mean_l1"], 2)} for k, v in sparsity.items()])
    _save_table(df_sp, "tabel_sparsity")

    # ---------------- Tabel konfigurasi (Bagian II)
    df_cfg = hyperparam_table(models)
    _save_table(df_cfg, "tabel_konfigurasi_bagian_II")

    results = {"mlp_test_acc": test_acc, "recon": rec, "fid": fid, "latency_ms": lat,
               "train_time_per_epoch_s": {k: t_epoch(k) for k in names}, "stability_cv": stab,
               "stability_cv_D": stab_d, "sparsity": sparsity, "orthogonality": ortho,
               "n_fid": C.N_FID, "epochs": C.EPOCHS, "device": torch.cuda.get_device_name(0)
               if device.type == "cuda" else "cpu"}
    save_json(results, os.path.join(C.RESULT_DIR, "all_results.json"))
    print(f"\nMLP test accuracy: {test_acc:.4f} (syarat >= {C.MLP_MIN_ACC:.0%})")
    print(f"Semua tabel tersimpan di: {C.RESULT_DIR}")
    ctx.update(results=results, tables={"main": df_main, "mlp": df_mlp, "ortho": df_ortho,
                                        "sparsity": df_sp, "config": df_cfg})
    return ctx


def _rec_cols(r):
    return {"Test MSE": round(r["mse"], 5), "Test PSNR (dB)": round(r["psnr"], 2), "Test SSIM": round(r["ssim"], 4)}


def hyperparam_table(models):
    gan_d = {"vanilla_gan": "D: 784-512-256-1 (LeakyReLU 0.2)",
             "dcgan": "D: Conv(1-64-128, stride 2) + BN, LeakyReLU 0.2, Conv 7x7 -> 1"}
    rows = [
        ("MLP Classifier", "mlp", "784-256-64-10 (Dropout 0.2)", C.LATENT_DIM, "ReLU / Softmax", "Cross-Entropy",
         "Adam", C.LR["mlp"]),
        ("Frozen Latent Decoder", "frozen_decoder", "64-256-784 (input z_MLP beku)", C.LATENT_DIM, "ReLU / Sigmoid",
         "MSE", "Adam", C.LR["frozen_decoder"]),
        ("Vanilla AE", "vanilla_ae", "Enc 784-256-64, Dec 64-256-784", C.LATENT_DIM, "ReLU / Sigmoid", "MSE",
         "Adam", C.LR["vanilla_ae"]),
        ("Sparse AE", "sparse_ae", "Enc 784-256-64, Dec 64-256-784", C.LATENT_DIM, "ReLU / Sigmoid",
         f"MSE + {C.SPARSE_LAMBDA:g}*||h||_1", "Adam", C.LR["sparse_ae"]),
        ("VAE", "vae", "Enc 784-256-(mu,logvar 64), Dec 64-256-784", C.LATENT_DIM, "ReLU / Sigmoid",
         "BCE + KL(q(z|x)||N(0,I))", "Adam", C.LR["vae"]),
        ("Vanilla GAN", "vanilla_gan", "G: 100-256-512-1024-784; " + gan_d["vanilla_gan"], C.NOISE_DIM,
         "LeakyReLU 0.2 / Tanh", "BCE minimax (non-saturating G)", f"Adam b={C.GAN_BETAS}", C.LR["vanilla_gan"]),
        ("DCGAN", "dcgan", "G: ConvT 100-256(7x7)-128(14x14)-1(28x28) + BN; " + gan_d["dcgan"], C.NOISE_DIM,
         "ReLU / Tanh", "BCE minimax (non-saturating G)", f"Adam b={C.GAN_BETAS}", C.LR["dcgan"]),
        ("DDPM", "ddpm", f"U-Net (ch {C.DDPM_BASE_CH}-{2 * C.DDPM_BASE_CH}, 28-14-7), T={C.DDPM_T}, "
         f"beta linear {C.DDPM_BETA_START}-{C.DDPM_BETA_END}, EMA {C.EMA_DECAY}", "28x28", "SiLU, GroupNorm",
         "MSE noise prediction (L_simple)", "Adam", C.LR["ddpm"]),
    ]
    return pd.DataFrame([{"Model": a, "Arsitektur": b, "Dim Laten": d, "Aktivasi": e, "Loss": f,
                          "Optimizer": g, "Learning Rate": lr, "Epoch": C.EPOCHS[k], "Batch": C.BATCH_SIZE,
                          "Jumlah Parameter (GAN: Generator)": f"{count_params(models[k]):,}"}
                         for a, k, b, d, e, f, g, lr in rows])

Overwriting src/evaluate.py


In [19]:
%%writefile src/figures.py
"""Lima figur wajib laporan (Bagian V). Disimpan PNG 300 dpi di outputs/figures."""
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from torchvision.utils import make_grid

from . import config as C
from . import experiments as X
from .data import pick_one_per_class

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "axes.titlesize": 11,
                     "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight"})


def _np(imgs, nrow):
    grid = make_grid(imgs.detach().cpu().clamp(0, 1), nrow=nrow, padding=2, pad_value=1.0)
    return grid.permute(1, 2, 0).numpy()


def _save(fig, name):
    path = os.path.join(C.FIG_DIR, name)
    fig.savefig(path)
    print(f"Tersimpan: {path}")
    plt.show()
    plt.close(fig)


def _row_figure(rows, title, ncols, fname, col_labels=None):
    """Figur bertingkat: setiap baris = satu strip citra dengan label di sebelah kiri."""
    fig, axes = plt.subplots(len(rows), 1, figsize=(1.0 * ncols + 2.6, 1.05 * len(rows) + 0.8))
    axes = np.atleast_1d(axes)
    for ax, (label, imgs) in zip(axes, rows):
        ax.imshow(_np(imgs, ncols), cmap="gray", vmin=0, vmax=1)
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_visible(False)
        ax.set_ylabel(label, rotation=0, ha="right", va="center", fontsize=9)
    if col_labels is not None:
        w = 28 + 2
        axes[0].set_xticks([2 + w * i + 14 for i in range(ncols)])
        axes[0].set_xticklabels(col_labels, fontsize=7.5)
        axes[0].xaxis.set_ticks_position("top")
        axes[0].tick_params(length=0)
    fig.suptitle(title, fontsize=12, fontweight="bold")
    fig.tight_layout()
    _save(fig, fname)


# ------------------------------------------------------------------ Figur 1
@torch.no_grad()
def figure1_reconstruction(ctx):
    m, dev = ctx["models"], ctx["device"]
    x, _ = pick_one_per_class(8)
    x = x.to(dev)
    vae = m["vae"]
    rows = [("Ground Truth", x),
            ("Vanilla AE", m["vanilla_ae"](x)[0]),
            ("Sparse AE", m["sparse_ae"](x)[0]),
            ("VAE", vae.decode(vae.encode(x)[0]))]
    _row_figure(rows, "Figur 1. Rekonstruksi Side-by-Side pada Test Set", 8, "figur1_rekonstruksi.png",
                col_labels=C.CLASS_NAMES[:8])


# ------------------------------------------------------------------ Figur 2
def figure2_sampling(ctx):
    s = ctx["samples"]
    panels = [("A. VAE  z~N(0,I)", s["vae"]), ("B. Vanilla GAN  z~N(0,I)", s["vanilla_gan"]),
              ("C. DCGAN  z~N(0,I)", s["dcgan"]), ("D. DDPM  x_T~N(0,I)", s["ddpm"])]
    fig, axes = plt.subplots(1, 4, figsize=(14, 4))
    for ax, (t, imgs) in zip(axes, panels):
        ax.imshow(_np(imgs[:16], 4), cmap="gray", vmin=0, vmax=1)
        ax.set_title(t); ax.axis("off")
    fig.suptitle("Figur 2. Random Generative Sampling (kisi 4x4 per model)", fontsize=12, fontweight="bold")
    fig.tight_layout()
    _save(fig, "figur2_random_sampling.png")


# ------------------------------------------------------------------ Figur 3
@torch.no_grad()
def figure3_mlp_limitations(ctx):
    m, dev = ctx["models"], ctx["device"]
    clf, dec, ae = m["mlp"], m["frozen_decoder"], m["vanilla_ae"]
    x, _ = pick_one_per_class(10)
    x = x.to(dev)
    z = clf.features(x)
    top = X.topdown_from_class(clf)
    lo = top.flatten(1).min(1)[0].view(-1, 1, 1, 1)
    hi = top.flatten(1).max(1)[0].view(-1, 1, 1, 1)
    top_norm = (top - lo) / (hi - lo + 1e-8)
    rnd = X.mlp_prior_sampling(dec, ctx["mu_z"], ctx["sd_z"], 10, dev, seed=C.SEED + 7)
    rows = [("(1) Ground Truth", x),
            ("(2a) W^T dari one-hot y\n(skala asli)", top),
            ("(2b) W^T dari one-hot y\n(min-max, visibilitas)", top_norm),
            ("(2c) W^T dari z_MLP\n(latent transpose)", X.latent_transpose(clf, z)),
            ("(3a) Frozen Decoder\ndari z_MLP", dec(z)),
            ("(3b) Vanilla AE\ndari z_AE (pembanding)", ae(x)[0]),
            ("(4) Prior sampling\nz~N(mu_z, diag(sigma_z^2))", rnd)]
    _row_figure(rows, "Figur 3. Bukti Empiris Batasan Bobot & Latent Features MLP Classifier", 10,
                "figur3_batasan_mlp.png", col_labels=C.CLASS_NAMES)


# ------------------------------------------------------------------ Figur 4
@torch.no_grad()
def figure4_interpolation(ctx):
    m, dev = ctx["models"], ctx["device"]
    x, _ = pick_one_per_class(10)
    xa = x[C.INTERP_CLASS_A:C.INTERP_CLASS_A + 1].to(dev)
    xb = x[C.INTERP_CLASS_B:C.INTERP_CLASS_B + 1].to(dev)
    rows = [("Vanilla AE", X.interpolate_autoencoder(m["vanilla_ae"], xa, xb)),
            ("Sparse AE", X.interpolate_autoencoder(m["sparse_ae"], xa, xb)),
            ("VAE", X.interpolate_autoencoder(m["vae"], xa, xb))]
    for key, label in [("vanilla_gan", "Vanilla GAN"), ("dcgan", "DCGAN")]:
        za, zb, pa, pb = X.find_gan_endpoints(m[key], m["mlp"], dev)
        print(f"{label}: endpoint z_A P({C.CLASS_NAMES[C.INTERP_CLASS_A]})={pa:.3f}, "
              f"z_B P({C.CLASS_NAMES[C.INTERP_CLASS_B]})={pb:.3f}")
        rows.append((label, X.interpolate_gan(m[key], za, zb)))
    _row_figure(rows, f"Figur 4. Linear Latent Interpolation z(a)=(1-a)z_A + a z_B "
                      f"({C.CLASS_NAMES[C.INTERP_CLASS_A]} -> {C.CLASS_NAMES[C.INTERP_CLASS_B]})",
                len(C.INTERP_ALPHAS), "figur4_interpolasi.png",
                col_labels=[f"a={a:.1f}" for a in C.INTERP_ALPHAS])


# ------------------------------------------------------------------ Figur 5
def _smooth(v, k=50):
    v = np.asarray(v, dtype=float)
    if len(v) < k:
        return v
    return np.convolve(v, np.ones(k) / k, mode="valid")


def figure5_losses_and_trajectory(ctx):
    L = ctx["logs"]
    fig = plt.figure(figsize=(15, 11.5))
    gs = fig.add_gridspec(3, 3, height_ratios=[1, 1, 1.0], hspace=0.45, wspace=0.28)

    def ep(v):
        return np.arange(1, len(v) + 1)

    ax = fig.add_subplot(gs[0, 0])
    ax.plot(ep(L["vanilla_ae"]["train_mse"]), L["vanilla_ae"]["train_mse"], marker="o", ms=3, label="train MSE")
    ax.plot(ep(L["vanilla_ae"]["val_mse"]), L["vanilla_ae"]["val_mse"], "--", marker="o", ms=3, label="val MSE")
    ax.set_title("(a) Vanilla AE"); ax.set_xlabel("epoch"); ax.set_ylabel("MSE"); ax.legend()

    ax = fig.add_subplot(gs[0, 1])
    ax.plot(ep(L["sparse_ae"]["train_loss"]), L["sparse_ae"]["train_loss"], marker="o", ms=3, label="total (MSE + lambda L1)")
    ax.plot(ep(L["sparse_ae"]["train_mse"]), L["sparse_ae"]["train_mse"], marker="o", ms=3, label="train MSE")
    ax.plot(ep(L["sparse_ae"]["val_mse"]), L["sparse_ae"]["val_mse"], "--", marker="o", ms=3, label="val MSE")
    ax.set_title("(b) Sparse AE"); ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend()

    ax = fig.add_subplot(gs[0, 2])
    ax.plot(ep(L["vae"]["train_loss"]), L["vae"]["train_loss"], marker="o", ms=3, label="-ELBO train")
    ax.plot(ep(L["vae"]["val_loss"]), L["vae"]["val_loss"], "--", marker="o", ms=3, label="-ELBO val")
    ax.plot(ep(L["vae"]["train_bce"]), L["vae"]["train_bce"], marker="o", ms=3, label="BCE")
    ax2 = ax.twinx()
    ax2.plot(ep(L["vae"]["train_kld"]), L["vae"]["train_kld"], color="tab:red", marker="o", ms=3, label="KL (sumbu kanan)")
    ax2.set_ylabel("KL")
    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=8)
    ax.set_title("(c) VAE"); ax.set_xlabel("epoch"); ax.set_ylabel("loss / citra")

    for col, (key, label) in enumerate([("vanilla_gan", "(d) Vanilla GAN"), ("dcgan", "(e) DCGAN")]):
        ax = fig.add_subplot(gs[1, col])
        it_per_ep = len(L[key]["iter_lossD"]) / max(len(L[key]["lossD"]), 1)
        for k_it, name, col in [("iter_lossD", "D", "tab:orange"), ("iter_lossG", "G", "tab:blue")]:
            v = L[key][k_it]
            ax.plot(np.arange(len(v)) / it_per_ep, v, color=col, alpha=0.15, lw=0.6)
            sm = _smooth(v)
            ax.plot(np.arange(len(sm)) / it_per_ep, sm, color=col, label=f"loss {name} (moving avg)")
        ax.set_title(label); ax.set_xlabel("epoch"); ax.set_ylabel("BCE loss"); ax.legend()

    ax = fig.add_subplot(gs[1, 2])
    ax.plot(ep(L["ddpm"]["train_loss"]), L["ddpm"]["train_loss"], marker="o", ms=3, label="train")
    ax.plot(ep(L["ddpm"]["val_loss"]), L["ddpm"]["val_loss"], "--", marker="o", ms=3, label="val")
    ax.set_title("(f) DDPM noise-prediction loss"); ax.set_xlabel("epoch"); ax.set_ylabel("MSE(eps)")
    ax.legend()

    traj = torch.load(os.path.join(C.SAMPLE_DIR, "ddpm_trajectory.pt"))
    steps = [s for s in C.DDPM_TRAJ_STEPS if s in traj]
    n = traj[steps[0]].size(0)
    n = min(n, 4)
    imgs = torch.stack([((traj[s][:n] + 1) / 2).clamp(0, 1) for s in steps], dim=1).flatten(0, 1)  # sampel x step
    ax = fig.add_subplot(gs[2, :])
    ax.imshow(_np(imgs, len(steps)), cmap="gray", vmin=0, vmax=1)
    w = 30
    ax.set_xticks([2 + w * i + 14 for i in range(len(steps))])
    ax.set_xticklabels([f"t={s}" for s in steps], fontsize=9)
    ax.set_yticks([]); ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(f"(g) DDPM reverse diffusion trajectory x_T -> x_0 (T={C.DDPM_T}, {n} sampel)")
    fig.suptitle("Figur 5. Training Loss Curves dan DDPM Denoising Trajectory", fontsize=13, fontweight="bold")
    _save(fig, "figur5_loss_dan_trajektori_ddpm.png")


def make_all_figures(ctx):
    os.makedirs(C.FIG_DIR, exist_ok=True)
    figure1_reconstruction(ctx)
    figure2_sampling(ctx)
    figure3_mlp_limitations(ctx)
    figure4_interpolation(ctx)
    figure5_losses_and_trajectory(ctx)

Overwriting src/figures.py


In [20]:
%%writefile src/run.py
"""Menjalankan seluruh pipeline dari terminal.

    python -m src.run --part A   # MLP + eksperimen batasan, Vanilla AE, Sparse AE, VAE, Vanilla GAN, DCGAN
    python -m src.run --part B   # DDPM (training + sampling)
    python -m src.run --part C   # evaluasi (tabel metrik) + 5 figur
    python -m src.run --part all
"""
import argparse

from . import data, evaluate, figures, train
from .utils import ensure_dirs, get_device


def part_a(loaders, device):
    clf, _ = train.train_mlp_classifier(loaders, device)
    train.train_frozen_decoder(clf, loaders, device)
    train.train_autoencoder(loaders, device, sparse=False)
    train.train_autoencoder(loaders, device, sparse=True)
    train.train_vae(loaders, device)
    train.train_gan(loaders, device, kind="vanilla_gan")
    train.train_gan(loaders, device, kind="dcgan")


def part_b(loaders, device):
    train.train_ddpm(loaders, device)
    train.generate_ddpm_samples(device)


def part_c(loaders, device):
    ctx = evaluate.run_evaluation(loaders, device)
    figures.make_all_figures(ctx)
    return ctx


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--part", choices=["A", "B", "C", "all"], default="all")
    args = p.parse_args()
    ensure_dirs()
    device = get_device()
    print("Device:", device)
    loaders = data.get_loaders()
    if args.part in ("A", "all"):
        part_a(loaders, device)
    if args.part in ("B", "all"):
        part_b(loaders, device)
    if args.part in ("C", "all"):
        part_c(loaders, device)


if __name__ == "__main__":
    main()

Overwriting src/run.py


## 2. Import & data

In [21]:
import sys
sys.path.insert(0, PROJECT_DIR)

from src import config as C, data, train, evaluate, figures
from src.utils import ensure_dirs, get_device

ensure_dirs()
device = get_device()
print("Device:", device, "| Output:", C.OUT_DIR)
loaders = data.get_loaders()

Device: cuda | Output: /content/drive/MyDrive/DL_Tugas_Sesi07/outputs
Split -> train: 56000 | val: 7000 | test: 7000 | batch_size: 128


## Bagian A — MLP Classifier, Autoencoder, VAE, GAN

In [22]:
clf, log_mlp = train.train_mlp_classifier(loaders, device)


===== mlp | params: 218,058 | epoch target: 20 | mulai: 0 =====
[mlp] epoch   1/20 | CE loss 0.6649 | val acc 0.8394 | 1.5s
[mlp] epoch   2/20 | CE loss 0.4376 | val acc 0.8564 | 2.3s
[mlp] epoch   3/20 | CE loss 0.3922 | val acc 0.8646 | 3.8s
[mlp] epoch   4/20 | CE loss 0.3690 | val acc 0.8673 | 2.5s
[mlp] epoch   5/20 | CE loss 0.3489 | val acc 0.8720 | 2.0s
[mlp] epoch   6/20 | CE loss 0.3337 | val acc 0.8739 | 1.8s
[mlp] epoch   7/20 | CE loss 0.3190 | val acc 0.8773 | 1.2s
[mlp] epoch   8/20 | CE loss 0.3128 | val acc 0.8807 | 1.3s
[mlp] epoch   9/20 | CE loss 0.3018 | val acc 0.8834 | 1.6s
[mlp] epoch  10/20 | CE loss 0.2937 | val acc 0.8869 | 1.4s
[mlp] epoch  11/20 | CE loss 0.2865 | val acc 0.8809 | 1.3s
[mlp] epoch  12/20 | CE loss 0.2817 | val acc 0.8879 | 1.2s
[mlp] epoch  13/20 | CE loss 0.2747 | val acc 0.8820 | 1.2s
[mlp] epoch  14/20 | CE loss 0.2663 | val acc 0.8949 | 1.2s
[mlp] epoch  15/20 | CE loss 0.2618 | val acc 0.8926 | 1.2s
[mlp] epoch  16/20 | CE loss 0.2577

In [23]:
frozen_dec, log_fd = train.train_frozen_decoder(clf, loaders, device)


===== frozen_decoder | params: 218,128 | epoch target: 15 | mulai: 0 =====
[frozen_decoder] epoch   1/15 | MSE 0.04071 | val MSE 0.03192 | 1.3s
[frozen_decoder] epoch   2/15 | MSE 0.02948 | val MSE 0.02833 | 1.2s
[frozen_decoder] epoch   3/15 | MSE 0.02697 | val MSE 0.02655 | 1.1s
[frozen_decoder] epoch   4/15 | MSE 0.02558 | val MSE 0.02547 | 1.1s
[frozen_decoder] epoch   5/15 | MSE 0.02465 | val MSE 0.02486 | 1.1s
[frozen_decoder] epoch   6/15 | MSE 0.02397 | val MSE 0.02437 | 1.1s
[frozen_decoder] epoch   7/15 | MSE 0.02343 | val MSE 0.02376 | 1.1s
[frozen_decoder] epoch   8/15 | MSE 0.02302 | val MSE 0.02338 | 1.1s
[frozen_decoder] epoch   9/15 | MSE 0.02264 | val MSE 0.02303 | 1.3s
[frozen_decoder] epoch  10/15 | MSE 0.02231 | val MSE 0.02279 | 2.1s
[frozen_decoder] epoch  11/15 | MSE 0.02206 | val MSE 0.02253 | 1.6s
[frozen_decoder] epoch  12/15 | MSE 0.02183 | val MSE 0.02233 | 1.1s
[frozen_decoder] epoch  13/15 | MSE 0.02159 | val MSE 0.02219 | 1.1s
[frozen_decoder] epoch  14/

In [24]:
ae, log_ae = train.train_autoencoder(loaders, device, sparse=False)


===== vanilla_ae | params: 435,536 | epoch target: 20 | mulai: 0 =====
[vanilla_ae] epoch   1/20 | loss 0.03199 | MSE 0.03199 | L1(h) 218.80 | val MSE 0.01854 | 1.4s
[vanilla_ae] epoch   2/20 | loss 0.01641 | MSE 0.01641 | L1(h) 227.43 | val MSE 0.01508 | 1.4s
[vanilla_ae] epoch   3/20 | loss 0.01380 | MSE 0.01380 | L1(h) 235.63 | val MSE 0.01307 | 1.4s
[vanilla_ae] epoch   4/20 | loss 0.01227 | MSE 0.01227 | L1(h) 241.60 | val MSE 0.01194 | 1.7s
[vanilla_ae] epoch   5/20 | loss 0.01121 | MSE 0.01121 | L1(h) 249.39 | val MSE 0.01107 | 1.7s
[vanilla_ae] epoch   6/20 | loss 0.01048 | MSE 0.01048 | L1(h) 249.32 | val MSE 0.01040 | 1.4s
[vanilla_ae] epoch   7/20 | loss 0.00984 | MSE 0.00984 | L1(h) 254.38 | val MSE 0.00989 | 1.3s
[vanilla_ae] epoch   8/20 | loss 0.00936 | MSE 0.00936 | L1(h) 257.41 | val MSE 0.00954 | 1.5s
[vanilla_ae] epoch   9/20 | loss 0.00897 | MSE 0.00897 | L1(h) 258.76 | val MSE 0.00903 | 1.3s
[vanilla_ae] epoch  10/20 | loss 0.00862 | MSE 0.00862 | L1(h) 258.12 | v

In [25]:
sae, log_sae = train.train_autoencoder(loaders, device, sparse=True)


===== sparse_ae | params: 435,536 | epoch target: 20 | mulai: 0 =====
[sparse_ae] epoch   1/20 | loss 0.04232 | MSE 0.04065 | L1(h) 16.70 | val MSE 0.02396 | 1.5s
[sparse_ae] epoch   2/20 | loss 0.02280 | MSE 0.02166 | L1(h) 11.40 | val MSE 0.01994 | 1.7s
[sparse_ae] epoch   3/20 | loss 0.01964 | MSE 0.01860 | L1(h) 10.45 | val MSE 0.01804 | 1.7s
[sparse_ae] epoch   4/20 | loss 0.01817 | MSE 0.01719 | L1(h) 9.83 | val MSE 0.01702 | 1.4s
[sparse_ae] epoch   5/20 | loss 0.01717 | MSE 0.01622 | L1(h) 9.49 | val MSE 0.01615 | 1.4s
[sparse_ae] epoch   6/20 | loss 0.01639 | MSE 0.01547 | L1(h) 9.16 | val MSE 0.01543 | 1.4s
[sparse_ae] epoch   7/20 | loss 0.01576 | MSE 0.01488 | L1(h) 8.89 | val MSE 0.01500 | 1.4s
[sparse_ae] epoch   8/20 | loss 0.01526 | MSE 0.01440 | L1(h) 8.67 | val MSE 0.01458 | 1.4s
[sparse_ae] epoch   9/20 | loss 0.01484 | MSE 0.01399 | L1(h) 8.49 | val MSE 0.01427 | 1.4s
[sparse_ae] epoch  10/20 | loss 0.01446 | MSE 0.01361 | L1(h) 8.47 | val MSE 0.01372 | 1.6s
[spars

In [26]:
vae, log_vae = train.train_vae(loaders, device)


===== vae | params: 451,984 | epoch target: 20 | mulai: 0 =====
[vae] epoch   1/20 | -ELBO 299.95 (BCE 285.31 + KL 14.64) | val -ELBO 273.43 | val MSE 0.02643 | 1.9s
[vae] epoch   2/20 | -ELBO 266.49 (BCE 250.66 + KL 15.83) | val -ELBO 262.54 | val MSE 0.02330 | 1.6s
[vae] epoch   3/20 | -ELBO 257.46 (BCE 242.39 + KL 15.07) | val -ELBO 255.21 | val MSE 0.01979 | 1.6s
[vae] epoch   4/20 | -ELBO 252.42 (BCE 237.80 + KL 14.62) | val -ELBO 251.84 | val MSE 0.01859 | 1.7s
[vae] epoch   5/20 | -ELBO 249.61 (BCE 235.04 + KL 14.57) | val -ELBO 249.62 | val MSE 0.01764 | 1.7s
[vae] epoch   6/20 | -ELBO 247.91 (BCE 233.23 + KL 14.68) | val -ELBO 247.96 | val MSE 0.01655 | 1.8s
[vae] epoch   7/20 | -ELBO 246.66 (BCE 231.76 + KL 14.90) | val -ELBO 247.05 | val MSE 0.01585 | 2.1s
[vae] epoch   8/20 | -ELBO 245.64 (BCE 230.60 + KL 15.05) | val -ELBO 246.44 | val MSE 0.01532 | 1.8s
[vae] epoch   9/20 | -ELBO 244.92 (BCE 229.67 + KL 15.25) | val -ELBO 245.71 | val MSE 0.01504 | 1.6s
[vae] epoch  10/2

In [27]:
gan_G, log_gan = train.train_gan(loaders, device, kind="vanilla_gan")


[vanilla_gan] params D: 533,505

===== vanilla_gan | params: 1,486,352 | epoch target: 50 | mulai: 0 =====
[vanilla_gan] epoch   1/50 | loss D 0.9867 | loss G 1.1240 | D(x) 0.772 | D(G(z)) 0.481 | 3.4s
[vanilla_gan] epoch   2/50 | loss D 0.9731 | loss G 1.7888 | D(x) 0.687 | D(G(z)) 0.376 | 2.7s
[vanilla_gan] epoch   3/50 | loss D 0.9034 | loss G 1.9218 | D(x) 0.701 | D(G(z)) 0.325 | 2.7s
[vanilla_gan] epoch   4/50 | loss D 1.1080 | loss G 1.4885 | D(x) 0.624 | D(G(z)) 0.395 | 2.8s
[vanilla_gan] epoch   5/50 | loss D 1.1074 | loss G 1.4271 | D(x) 0.619 | D(G(z)) 0.393 | 3.0s
[vanilla_gan] epoch   6/50 | loss D 1.1292 | loss G 1.3507 | D(x) 0.609 | D(G(z)) 0.399 | 3.2s
[vanilla_gan] epoch   7/50 | loss D 1.1488 | loss G 1.2902 | D(x) 0.603 | D(G(z)) 0.405 | 2.8s
[vanilla_gan] epoch   8/50 | loss D 1.1424 | loss G 1.2995 | D(x) 0.604 | D(G(z)) 0.402 | 2.8s
[vanilla_gan] epoch   9/50 | loss D 1.1729 | loss G 1.2196 | D(x) 0.595 | D(G(z)) 0.411 | 2.8s
[vanilla_gan] epoch  10/50 | loss D 1

In [28]:
dcgan_G, log_dcgan = train.train_gan(loaders, device, kind="dcgan")


[dcgan] params D: 138,689

===== dcgan | params: 1,781,505 | epoch target: 20 | mulai: 0 =====
[dcgan] epoch   1/20 | loss D 0.6492 | loss G 1.8296 | D(x) 0.753 | D(G(z)) 0.254 | 18.2s
[dcgan] epoch   2/20 | loss D 0.6160 | loss G 1.8387 | D(x) 0.762 | D(G(z)) 0.234 | 17.3s
[dcgan] epoch   3/20 | loss D 0.8501 | loss G 1.4543 | D(x) 0.684 | D(G(z)) 0.314 | 16.8s
[dcgan] epoch   4/20 | loss D 0.9356 | loss G 1.3273 | D(x) 0.656 | D(G(z)) 0.342 | 16.9s
[dcgan] epoch   5/20 | loss D 0.9535 | loss G 1.3107 | D(x) 0.652 | D(G(z)) 0.348 | 17.2s
[dcgan] epoch   6/20 | loss D 0.9795 | loss G 1.2986 | D(x) 0.646 | D(G(z)) 0.354 | 17.2s
[dcgan] epoch   7/20 | loss D 0.9986 | loss G 1.2780 | D(x) 0.640 | D(G(z)) 0.360 | 17.1s
[dcgan] epoch   8/20 | loss D 1.0113 | loss G 1.2601 | D(x) 0.635 | D(G(z)) 0.364 | 16.9s
[dcgan] epoch   9/20 | loss D 1.0282 | loss G 1.2503 | D(x) 0.631 | D(G(z)) 0.368 | 17.1s
[dcgan] epoch  10/20 | loss D 1.0612 | loss G 1.2363 | D(x) 0.625 | D(G(z)) 0.374 | 17.1s
[dcg

## Bagian B — DDPM
Bisa dijalankan di akun Colab kedua secara bersamaan (jalankan bagian 0, 1, 2, lalu sel di bawah).

In [29]:
ddpm_ema, log_ddpm = train.train_ddpm(loaders, device)


===== ddpm | params: 3,098,561 | epoch target: 20 | mulai: 0 =====
[ddpm] T = 1000 | mixed precision (AMP): True
[ddpm] epoch   1/20 | noise MSE 0.08615 | val 0.05193 | 47.2s | perkiraan sisa 14.9 menit
[ddpm] epoch   2/20 | noise MSE 0.05153 | val 0.04668 | 46.7s | perkiraan sisa 14.0 menit
[ddpm] epoch   3/20 | noise MSE 0.04675 | val 0.04331 | 46.8s | perkiraan sisa 13.2 menit
[ddpm] epoch   4/20 | noise MSE 0.04447 | val 0.04365 | 46.7s | perkiraan sisa 12.5 menit
[ddpm] epoch   5/20 | noise MSE 0.04341 | val 0.04254 | 46.8s | perkiraan sisa 11.7 menit
[ddpm] epoch   6/20 | noise MSE 0.04205 | val 0.04109 | 46.7s | perkiraan sisa 10.9 menit
[ddpm] epoch   7/20 | noise MSE 0.04148 | val 0.03956 | 46.8s | perkiraan sisa 10.1 menit
[ddpm] epoch   8/20 | noise MSE 0.04113 | val 0.03802 | 46.6s | perkiraan sisa 9.3 menit
[ddpm] epoch   9/20 | noise MSE 0.04063 | val 0.03997 | 46.8s | perkiraan sisa 8.6 menit
[ddpm] epoch  10/20 | noise MSE 0.03951 | val 0.03741 | 46.7s | perkiraan sisa

In [30]:
ddpm_samples = train.generate_ddpm_samples(device)

[ddpm] trajektori denoising tersimpan.
[ddpm] chunk 1/4 (500 sampel) selesai dalam 150.2s
[ddpm] chunk 2/4 (500 sampel) selesai dalam 149.8s
[ddpm] chunk 3/4 (500 sampel) selesai dalam 150.0s
[ddpm] chunk 4/4 (500 sampel) selesai dalam 150.2s
[ddpm] 2000 sampel FID tersimpan -> /content/drive/MyDrive/DL_Tugas_Sesi07/outputs/samples/ddpm_fid_samples.pt


## Bagian C — Evaluasi Test Set & Figur
Jika DDPM dilatih di akun lain, jalankan dulu `drive.mount('/content/drive', force_remount=True)`.

In [ ]:
from IPython.display import display

ctx = evaluate.run_evaluation(loaders, device)
for k in ["main", "mlp", "ortho", "sparsity", "config"]:
    display(ctx["tables"][k])

Menghitung MSE / PSNR / SSIM pada Test Set ...
Membuat 2000 sampel per model generatif ...
[ddpm] sampel FID sudah ada: /content/drive/MyDrive/DL_Tugas_Sesi07/outputs/samples/ddpm_fid_samples.pt
Menghitung FID (Inception-v3) ...
  FID vae         : 97.18
  FID vanilla_gan : 128.59
  FID dcgan       : 30.78
  FID ddpm        : 19.68
  FID mlp_prior   : 152.59
Mengukur inference latency (batch 64) ...


In [ ]:
figures.make_all_figures(ctx)

In [ ]:
# simpan salinan notebook ke folder repo
!cp "/content/drive/MyDrive/Colab Notebooks/DL_Tugas_Sesi07.ipynb" notebook/ 2>/dev/null || echo "Salin notebook manual: File > Download .ipynb"
!ls -R outputs | head -50